┌─────────────────────────────────────────────────────────┐
│                    DATA LAYER                           │
│  (Your existing pipeline — historical + live streaming) │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  FEATURE LAYER                          │
│  (Alignment, resampling, all engineered features)       │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  SIGNAL LAYER                           │
│  (Individual setups A/B/C/D — each independently)      │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  RISK LAYER                             │
│  (Position sizing, max drawdown, correlation limits)    │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│               EXECUTION LAYER                           │
│  (Order routing, slippage, Binance API)                 │
└─────────────────────────────────────────────────────────┘

In [ ]:
from orderflow import futures_agg_trades, get_oi, get_mark_price_klines, spot_agg_trades, get_premium_index_klines
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor

symbol = "BTCUSDT"
start = pd.Timestamp("2026-06-10 00:00")
end   = pd.Timestamp("2026-06-11 00:00")
timeframe = "5min"

# Funding rate requires pre-warming hence the start - pd.Timedelta(days=1))
dates = [d.strftime("%Y-%m-%d")
         for d in pd.date_range((start - pd.Timedelta(days=1)).normalize(), end.normalize(), freq="D")]

streams = {
    "fut_trades": futures_agg_trades,
    "oi":         get_oi,
    "context":    get_mark_price_klines,
    "spot":       spot_agg_trades,
    "premium":    get_premium_index_klines,
}
tasks = [(name, fn, d) for name, fn in streams.items() for d in dates]
frames = {name: [] for name in streams}
with ThreadPoolExecutor(max_workers=min(32, len(tasks))) as ex:
    for name, df in ex.map(lambda t: (t[0], t[1](symbol, t[2])), tasks):
        frames[name].append(df)

fut_trades = pd.concat(frames["fut_trades"]).sort_index()
oi         = pd.concat(frames["oi"]).sort_index()
context    = pd.concat(frames["context"]).sort_index()
spot       = pd.concat(frames["spot"]).sort_index()
premium    = pd.concat(frames["premium"]).sort_index()

ohlc = context.resample(timeframe).agg({"open": "first", "high": "max", "low": "min", "close": "last"})

fut_cvd  = fut_trades["volume_delta"].resample(timeframe).sum().cumsum()
spot_cvd = spot["volume_delta"].resample(timeframe).sum().cumsum()

I = 0.0001
step = premium.index.to_series().diff().median()
n = int(pd.Timedelta("8h") / step)
w = np.arange(1, n + 1)
p_avg = premium["close"].rolling(n).apply(lambda x: x @ w / w.sum(), raw=True)
funding_rate = p_avg + (I - p_avg).clip(-0.0005, 0.0005)
funding_annualized = (funding_rate * 3 * 365).resample(timeframe).last()

oi_tf = oi["sum_open_interest"].resample(timeframe).last()

In [2]:
combined = pd.concat([
    fut_cvd.rename("fut_cumulative_volume_delta"),
    spot_cvd.rename("spot_cumulative_volume_delta"),
    ohlc,
    oi_tf.rename("sum_open_interest"),
    funding_annualized.rename("funding_rate_annualized"),
], axis=1)

combined = combined.loc[start.floor(timeframe):end]

for col in ["fut_cumulative_volume_delta", "spot_cumulative_volume_delta"]:
    combined[col] = combined[col] - combined[col].dropna().iloc[0]

combined

,fut_cumulative_volume_delta,spot_cumulative_volume_delta,open,high,low,close,sum_open_interest,funding_rate_annualized
2026-06-10 00:00:00,0.000,0.00000,61695.279174,61815.563630,61672.700000,61804.300000,99232.019,-0.001457
2026-06-10 00:05:00,-10.549,18.51797,61804.300000,61864.000000,61730.100000,61819.300000,99256.810,-0.001286
2026-06-10 00:10:00,-74.106,11.56108,61819.300000,61824.600000,61710.761435,61805.898746,99281.444,-0.002334
2026-06-10 00:15:00,-104.551,3.63009,61806.134181,61815.356543,61710.600000,61789.590360,99336.711,-0.003869
2026-06-10 00:20:00,-142.874,-3.59973,61790.245471,61790.245471,61691.600000,61755.162196,99360.774,-0.006258
...,...,...,...,...,...,...,...,...
2026-06-10 23:40:00,368.094,-563.50964,61463.204746,61582.100000,61449.187333,61516.300000,99266.192,-0.021816
2026-06-10 23:45:00,311.226,-574.40859,61517.200000,61522.100000,61458.900000,61464.515832,99273.907,-0.020647
2026-06-10 23:50:00,454.134,-564.71572,61464.087964,61557.046507,61448.333855,61515.700000,99269.779,-0.016745
2026-06-10 23:55:00,450.917,-562.99422,61515.900000,61523.000000,61476.200000,61489.482674,99350.372,-0.015277


In [3]:
from orderflow import build_order_flow_chart

my_layout = [
    {"type": "candlestick", "title": "Price Action", "name": "Candlestick", "y_title": "Price (USDT)"},
    {"type": "line", "title": "Open Interest", "column": "sum_open_interest", "name": "OI Coins", "color": "purple", "y_title": "OI"},
    {"type": "delta_bars", "title": "Funding Rate", "column": "funding_rate_annualized", "name": "Funding", "color": "orange", "y_title": "Rate"},
    {"type": "delta_bars", "title": "Spot CVD", "column": "spot_cumulative_volume_delta", "name": "Spot Delta", "y_title": "Delta"},
    {"type": "delta_bars", "title": "Futures CVD", "column": "fut_cumulative_volume_delta", "name": "Futures Delta", "y_title": "Delta"}
]

fig = build_order_flow_chart(combined, f"{start:%Y-%m-%d %H:%M} → {end:%Y-%m-%d %H:%M}", config=my_layout, timeframe=timeframe)
fig.show()